# ODI Complaints Cleaning

This notebook is where we use the EDA findings to create cleaning decisions for the complaints data.

Keep the work layered and inspectable so we can see what changes, why it changes, and how it affects the first modeling tables.

## 1. Setup And Ground Rules

Importing the basic tooling and making the notebook display rules explicit.

In [ ]:
# Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

# Preferred pandas and seaborn display parameters
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

## 2. Load The Working Dataset

Load the combined processed complaints file that came out of initial setup.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

COMBINED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'odi_complaints_combined.parquet'

raw_df = pd.read_parquet(COMBINED_PATH)
raw_df = raw_df.drop(columns=['source_zip', 'source_file'], errors='ignore')

print("Loaded:", COMBINED_PATH.name)
print("Shape:", raw_df.shape)

## 3. Source Data Notes

Before changing anything, pull in the parts of `CMPL.txt` with useful info for cleaning to keep the notebook anchored to the source documentation.

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.schema_checks import get_schema_spec

spec = get_schema_spec('complaints')
schema_df = pd.DataFrame(spec['fields'])
desc_map = schema_df.set_index('name')['description'].to_dict()

focus_cols = [
    'yeartxt', 'faildate', 'datea', 'ldate', 'miles', 'injured', 'deaths',
    'veh_speed', 'vin', 'purch_dt', 'manuf_dt', 'state', 'dealer_state',
    'prod_type', 'cdescr'
]

schema_view = schema_df.loc[
    schema_df['name'].isin(focus_cols),
    ['name', 'type', 'size', 'description']
].sort_values('name')

change_notes = pd.DataFrame([
    {'note': 'May-Jun 2021 file update', 'implication': 'Previously blank Y/N fields may now appear as N'},
    {'note': 'May-Jun 2021 file update', 'implication': 'Previously blank numeric fields may now appear as 0'},
    {'note': 'May-Jun 2021 file update', 'implication': 'Manufacturer, make, model, and component values may reflect newer cleanup over time'}
])

print("Schema overview of major features:")
display(schema_view)

print("\nDocumented schema changes:")
display(change_notes)

## 4. Modeling Context

Cleaning decisions can't be generic here, we have three different models for our project tasks with different requirements.

This section solidifies the modeling context first so the later rules are chosen for a reason.

In [ ]:
model_plan = pd.DataFrame([
    {
        'model': 'severity ranking model',
        'unit_of_analysis': 'complaint case keyed by odino',
        'time_anchor': 'ldate',
        'initial_cohort': "prod_type == 'V'",
        'target_or_output': 'primary high-risk flag from deaths, injured, fire, and crash',
        'baseline': 'logistic regression',
        'strong_candidate': 'CatBoostClassifier'
    },
    {
        'model': 'component category model',
        'unit_of_analysis': 'complaint-component row',
        'time_anchor': 'ldate',
        'initial_cohort': "prod_type == 'V'",
        'target_or_output': 'cleaned component group built from compdesc',
        'baseline': 'multinomial logistic regression',
        'strong_candidate': 'CatBoostClassifier'
    },
    {
        'model': 'NLP early warning watchlist',
        'unit_of_analysis': 'cohort/topic/component month',
        'time_anchor': 'ldate',
        'initial_cohort': "prod_type == 'V'",
        'target_or_output': 'monthly watchlist ranked by topic growth and complaint activity',
        'baseline': 'trend and spike rules',
        'strong_candidate': 'topic growth plus cohort monitoring'
    }
])

display(model_plan)

### 4.1 Vehicle Cohort Check

Confirm whether the vehicle cohort is large enough to justify a vehicle-first cleaning and modeling path.

There was large missingness in the data that was really structural based on product type rather than being bad data.

In [ ]:
prod_counts = raw_df['prod_type'].astype('string').value_counts(dropna=False).rename_axis('prod_type').reset_index(name='row_count')
prod_counts['row_pct'] = (prod_counts['row_count'] / len(raw_df) * 100).round(2)

vehicle_mask = raw_df['prod_type'].astype('string') == 'V'
vehicle_df = raw_df.loc[vehicle_mask].copy()

cohort_summary = pd.DataFrame([
    {
        'view': 'all rows',
        'rows': len(raw_df),
        'row_pct': 100.0,
        'unique_makes': raw_df['maketxt'].nunique(dropna=True),
        'unique_models': raw_df['modeltxt'].nunique(dropna=True),
        'fire_y_pct': round(float((raw_df['fire'].astype('string') == 'Y').mean() * 100), 2),
        'narrative_non_null_pct': round(float(raw_df['cdescr'].notna().mean() * 100), 2)
    },
    {
        'view': "prod_type='V'",
        'rows': len(vehicle_df),
        'row_pct': round(float(len(vehicle_df) / len(raw_df) * 100), 2),
        'unique_makes': vehicle_df['maketxt'].nunique(dropna=True),
        'unique_models': vehicle_df['modeltxt'].nunique(dropna=True),
        'fire_y_pct': round(float((vehicle_df['fire'].astype('string') == 'Y').mean() * 100), 2),
        'narrative_non_null_pct': round(float(vehicle_df['cdescr'].notna().mean() * 100), 2)
    }
])

core_cols = ['maketxt', 'modeltxt', 'yeartxt', 'state', 'miles', 'veh_speed', 'cdescr', 'fire', 'crash']
core_avail = pd.DataFrame({
    'all_rows_non_null_pct': (raw_df[core_cols].notna().mean() * 100).round(2),
    'vehicle_only_non_null_pct': (vehicle_df[core_cols].notna().mean() * 100).round(2)
}).reset_index().rename(columns={'index': 'column'})

print("Dataset product type makeup:")
display(prod_counts)

print("\nSummary of vehicle cohort vs full dataset:")
display(cohort_summary)

print("\nCoverage of major features in vehicle cohort vs full dataset: ")
display(core_avail)

### 4.2 Field Decision Matrix

This is the working decision table for the most important columns.

Want to explicitly define each field's meaning, issue pattern, and likely handling before we change anything.

In [ ]:
field_decisions = pd.DataFrame([
    {
        'field': 'cmplid',
        'schema_note': desc_map['cmplid'],
        'base_clean': 'keep as unique row key',
        'severity_model': 'not the case key',
        'component_model': 'row key only',
        'nlp_watchlist': 'not a feature',
        'current_call': 'settled',
        'note': ''
    },
    {
        'field': 'odino',
        'schema_note': desc_map['odino'],
        'base_clean': 'keep as complaint-case reference',
        'severity_model': 'case key',
        'component_model': 'reference only',
        'nlp_watchlist': 'case reference only',
        'current_call': 'settled',
        'note': ''
    },
    {
        'field': 'prod_type',
        'schema_note': desc_map['prod_type'],
        'base_clean': 'strip, uppercase, validate',
        'severity_model': 'start with V only',
        'component_model': 'start with V only',
        'nlp_watchlist': 'start with V only',
        'current_call': 'settled',
        'note': ''
    },
    {
        'field': 'maketxt',
        'schema_note': desc_map['maketxt'],
        'base_clean': 'strip and uppercase',
        'severity_model': 'baseline feature',
        'component_model': 'baseline feature',
        'nlp_watchlist': 'cohort field',
        'current_call': 'settled',
        'note': ''
    },
    {
        'field': 'modeltxt',
        'schema_note': desc_map['modeltxt'],
        'base_clean': 'strip and uppercase',
        'severity_model': 'baseline feature',
        'component_model': 'baseline feature',
        'nlp_watchlist': 'cohort field',
        'current_call': 'settled',
        'note': 'Make-aware cleanup may still help later'
    },
    {
        'field': 'compdesc',
        'schema_note': desc_map.get('compdesc', 'COMPONENT DESCRIPTION'),
        'base_clean': 'strip and uppercase',
        'severity_model': 'exclude from first baseline',
        'component_model': 'target source for cleaned component group',
        'nlp_watchlist': 'component bucket after grouping',
        'current_call': 'mostly settled',
        'note': 'Need a cleaned component grouping pass next'
    },
    {
        'field': 'yeartxt',
        'schema_note': desc_map['yeartxt'],
        'base_clean': 'convert to Int64 and null unknown',
        'severity_model': 'baseline feature',
        'component_model': 'baseline feature',
        'nlp_watchlist': 'cohort field',
        'current_call': 'settled',
        'note': 'Keep unknown-year flag'
    },
    {
        'field': 'ldate',
        'schema_note': desc_map['ldate'],
        'base_clean': 'preserve raw datetime',
        'severity_model': 'main time anchor',
        'component_model': 'main time anchor',
        'nlp_watchlist': 'main time anchor',
        'current_call': 'settled',
        'note': ''
    },
    {
        'field': 'faildate',
        'schema_note': desc_map['faildate'],
        'base_clean': 'preserve raw datetime and add trust flag',
        'severity_model': 'use only through safe derived lag features',
        'component_model': 'optional metadata',
        'nlp_watchlist': 'only if trusted',
        'current_call': 'settled',
        'note': "Trusted only when faildate <= ldate and fail_year >= model_year - 1"
    },
    {
        'field': 'datea',
        'schema_note': desc_map['datea'],
        'base_clean': 'preserve raw datetime',
        'severity_model': 'exclude from first baseline',
        'component_model': 'exclude from first baseline',
        'nlp_watchlist': 'exclude from first pass',
        'current_call': 'settled',
        'note': 'Keep for audit only'
    },
    {
        'field': 'miles',
        'schema_note': desc_map['miles'],
        'base_clean': 'convert to Int64, keep zero, flag highs',
        'severity_model': 'baseline feature plus missing flag',
        'component_model': 'baseline feature plus missing flag',
        'nlp_watchlist': 'cohort metadata',
        'current_call': 'settled',
        'note': 'No zero flag in v1'
    },
    {
        'field': 'veh_speed',
        'schema_note': desc_map['veh_speed'],
        'base_clean': 'convert to Int64, null 999, keep zero, flag highs',
        'severity_model': 'baseline feature plus missing flag',
        'component_model': 'baseline feature plus missing flag',
        'nlp_watchlist': 'cohort metadata',
        'current_call': 'settled',
        'note': 'No zero flag in v1 and >200 stays flagged'
    },
    {
        'field': 'state',
        'schema_note': desc_map['state'],
        'base_clean': 'strip, uppercase, validate',
        'severity_model': 'baseline feature',
        'component_model': 'baseline feature',
        'nlp_watchlist': 'regional cohort field',
        'current_call': 'settled',
        'note': ''
    },
    {
        'field': 'injured',
        'schema_note': desc_map['injured'],
        'base_clean': 'convert to Int64 and keep zero',
        'severity_model': 'label only, not a feature',
        'component_model': 'exclude from first baseline',
        'nlp_watchlist': 'severity summary later',
        'current_call': 'settled',
        'note': 'Primary severity label input'
    },
    {
        'field': 'deaths',
        'schema_note': desc_map['deaths'],
        'base_clean': 'convert to Int64 and keep zero',
        'severity_model': 'label only, not a feature',
        'component_model': 'exclude from first baseline',
        'nlp_watchlist': 'severity summary later',
        'current_call': 'settled',
        'note': 'Primary severity label input'
    },
    {
        'field': 'fire / crash',
        'schema_note': 'Y/N severity indicators',
        'base_clean': 'strip and uppercase',
        'severity_model': 'label only, not a feature',
        'component_model': 'exclude from first baseline',
        'nlp_watchlist': 'watchlist outcome signals',
        'current_call': 'settled',
        'note': 'Primary severity label input'
    },
    {
        'field': 'medical_attn / vehicles_towed_yn',
        'schema_note': 'Y/N severity-adjacent indicators',
        'base_clean': 'strip and uppercase',
        'severity_model': 'sensitivity label only',
        'component_model': 'exclude from first baseline',
        'nlp_watchlist': 'possible severity context later',
        'current_call': 'settled',
        'note': 'Not part of the primary severity label'
    },
    {
        'field': 'cdescr',
        'schema_note': desc_map['cdescr'],
        'base_clean': 'strip only',
        'severity_model': 'exclude from first structured baseline',
        'component_model': 'exclude from first structured baseline',
        'nlp_watchlist': 'main input field',
        'current_call': 'settled',
        'note': 'Real text prep stays in the NLP track'
    }
])

display(field_decisions)

## 5. Layered Cleaning Frames

The notebook is set up around layered data frames (the standardized frame, the boolean flags frame, and candidate features frame) rather than one destructive cleaning pass.

That keeps the raw values visible and makes it much easier to audit each decision.

In [ ]:
std_df = raw_df.copy()
flag_df = pd.DataFrame(index=raw_df.index)
candidate_df = None

YN_COLS = [
    'crash',
    'fire',
    'police_rpt_yn',
    'orig_owner_yn',
    'anti_brakes_yn',
    'cruise_cont_yn',
    'orig_equip_yn',
    'repaired_yn',
    'medical_attn',
    'vehicles_towed_yn'
]

UPPER_COLS = [
    'mfr_name',
    'maketxt',
    'modeltxt',
    'compdesc',
    'city',
    'state',
    'vin',
    'cmpl_type',
    'dealer_name',
    'dealer_city',
    'dealer_state',
    'prod_type'
] + YN_COLS

STRIP_ONLY_COLS = [
    'cdescr',
    'fuel_sys',
    'fuel_type',
    'trans_type',
    'drive_train',
    'dot',
    'tire_size',
    'loc_of_tire',
    'tire_fail_type',
    'dealer_zip',
    'dealer_tel'
]

INT_COLS = [
    'yeartxt',
    'injured',
    'deaths',
    'miles',
    'occurences',
    'num_cyls',
    'veh_speed'
]

DATE_STR_COLS = [
    'purch_dt',
    'manuf_dt'
]

VALID_PROD_TYPES = {
    'V',
    'T',
    'E',
    'C'
}

POSTAL_CODES = {
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'HI', 'ID', 'IL',
    'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MS', 'MO', 'MT',
    'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI',
    'SC', 'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'DC', 'PR',
    'VI', 'GU', 'AS', 'MP', 'FM', 'MH', 'PW', 'AE', 'AA', 'AP'
}

MISS_TAG = '__MISSING__'

# -----------------------------------------------------------------------------
# Small cleaning helpers
# -----------------------------------------------------------------------------
def strip_text(series):
    text = series.astype('string').str.strip()
    return text.replace({'': pd.NA})


def upper_text(series):
    return strip_text(series).str.upper()


def to_int(series):
    return pd.to_numeric(strip_text(series), errors='coerce').astype('Int64')


def to_dt(series):
    return pd.to_datetime(strip_text(series), format='%Y%m%d', errors='coerce')


# One helper here beats four copy-paste compare blocks
def compare_view(mask, cols, n=10):
    raw_view = raw_df.loc[mask, cols].add_prefix('raw_')
    std_view = std_df.loc[mask, cols].add_prefix('std_')
    cand_view = candidate_df.loc[mask, cols].add_prefix('cand_')
    return raw_view.join(std_view).join(cand_view).head(n)

### 5.1 Base Standardization

This step handles the safest shared standardization work first.

We normalize types and formatting without making stronger cleaning choices yet.

In [ ]:
text_rows = []
dtype_rows = []

# Format to uppercase and change dtype of all UPPER_COL features
for col in UPPER_COLS:
    if col not in std_df.columns:
        continue
    before = std_df[col].copy()
    std_df[col] = upper_text(std_df[col])
    changed = int(
        before.astype('string')
        .fillna(MISS_TAG)
        .ne(std_df[col].astype('string').fillna(MISS_TAG))
        .sum()
    )
    text_rows.append(
        {
            'column': col,
            'action': 'strip + upper',
            'changed_rows': changed,
            'null_after': int(std_df[col].isna().sum()),
        }
    )

# Strip whitespace and format all listed features
for col in STRIP_ONLY_COLS:
    if col not in std_df.columns:
        continue
    before = std_df[col].copy()
    std_df[col] = strip_text(std_df[col])
    changed = int(
        before.astype('string')
        .fillna(MISS_TAG)
        .ne(std_df[col].astype('string').fillna(MISS_TAG))
        .sum()
    )
    text_rows.append(
        {
            'column': col,
            'action': 'strip only',
            'changed_rows': changed,
            'null_after': int(std_df[col].isna().sum()),
        }
    )

# Change all columns that should be int
for col in INT_COLS:
    if col not in std_df.columns:
        continue
    before_dtype = str(std_df[col].dtype)
    before_non_null = int(std_df[col].notna().sum())
    std_df[col] = to_int(std_df[col])
    dtype_rows.append(
        {
            'column': col,
            'action': 'to Int64',
            'dtype_before': before_dtype,
            'dtype_after': str(std_df[col].dtype),
            'non_null_before': before_non_null,
            'non_null_after': int(std_df[col].notna().sum()),
        }
    )

# Format data columns as proper datetimes
for col in DATE_STR_COLS:
    if col not in std_df.columns:
        continue
    before_dtype = str(std_df[col].dtype)
    before_non_null = int(std_df[col].notna().sum())
    std_df[col] = to_dt(std_df[col])
    dtype_rows.append(
        {
            'column': col,
            'action': 'to datetime',
            'dtype_before': before_dtype,
            'dtype_after': str(std_df[col].dtype),
            'non_null_before': before_non_null,
            'non_null_after': int(std_df[col].notna().sum()),
        }
    )

text_log = pd.DataFrame(text_rows).sort_values(
    ['changed_rows', 'column'], ascending=[False, True]
)
dtype_log = pd.DataFrame(dtype_rows)

print("Changes from text actions: ")
display(text_log[text_log['changed_rows'] > 0])

print("\nChanges from data type changes: ")
display(dtype_log)

### 5.2 Issue Flags

Next we create flags for the values and patterns that need closer review.

These flags let us inspect suspicious records without prematurely deleting or overwriting them.

In [ ]:
current_year = pd.Timestamp.today().year
fail_year = std_df['faildate'].dt.year
known_model_year = std_df['yeartxt'].notna() & std_df['yeartxt'].ne(9999)

flag_df['flag_prod_type_bad'] = std_df['prod_type'].notna() & ~std_df['prod_type'].isin(VALID_PROD_TYPES)
flag_df['flag_year_unknown'] = std_df['yeartxt'].eq(9999)
flag_df['flag_year_out_of_range'] = std_df['yeartxt'].notna() & ~std_df['yeartxt'].between(1900, current_year + 1)
flag_df['flag_speed_999'] = std_df['veh_speed'].eq(999)
flag_df['flag_speed_high'] = std_df['veh_speed'].gt(200) & ~std_df['veh_speed'].eq(999)
flag_df['flag_miles_high'] = std_df['miles'].gt(500000)
flag_df['flag_injured_99'] = std_df['injured'].eq(99)
flag_df['flag_deaths_99'] = std_df['deaths'].eq(99)
flag_df['flag_state_bad'] = std_df['state'].notna() & ~std_df['state'].isin(POSTAL_CODES)
flag_df['flag_dealer_state_bad'] = std_df['dealer_state'].notna() & ~std_df['dealer_state'].isin(POSTAL_CODES)
flag_df['flag_vin_len_bad'] = std_df['vin'].notna() & std_df['vin'].str.len().ne(11)
flag_df['flag_fail_after_added'] = std_df['faildate'].notna() & std_df['datea'].notna() & (std_df['faildate'] > std_df['datea'])
flag_df['flag_fail_after_received'] = std_df['faildate'].notna() & std_df['ldate'].notna() & (std_df['faildate'] > std_df['ldate'])
flag_df['flag_added_before_received'] = std_df['datea'].notna() & std_df['ldate'].notna() & (std_df['datea'] < std_df['ldate'])
flag_df['flag_date_order_bad'] = flag_df[['flag_fail_after_added', 'flag_fail_after_received', 'flag_added_before_received']].any(axis=1)
flag_df['flag_fail_old_new_vehicle'] = fail_year.notna() & known_model_year & std_df['yeartxt'].ge(1990) & fail_year.lt(1990)
flag_df['flag_fail_pre_model'] = fail_year.notna() & known_model_year & fail_year.lt(std_df['yeartxt'] - 1)
flag_df['flag_fail_pre_model_far'] = fail_year.notna() & known_model_year & fail_year.lt(std_df['yeartxt'] - 5)

flag_summary = pd.DataFrame([
    {
        'flag': col,
        'row_count': int(flag_df[col].fillna(False).sum())
    }
    for col in flag_df.columns
])
flag_summary['row_pct'] = (flag_summary['row_count'] / len(std_df) * 100).round(4)
flag_summary = flag_summary.sort_values(['row_count', 'flag'], ascending=[False, True])

display(flag_summary)

### 5.3 2021 Changelog Watch

The complaint file format changed around 2021, especially for blank versus `0` and blank versus `N` behavior.

We check that pattern here so later cleaning does not mistake automatic system-filled values for observed values.

In [ ]:
zero_watch = (
    std_df.assign(received_year=std_df['ldate'].dt.year)
    .groupby('received_year')[['injured', 'deaths', 'miles', 'veh_speed']]
    .agg(lambda s: round(float((s == 0).mean() * 100), 2))
    .reset_index()
)

yn_watch = (
    std_df.assign(received_year=std_df['ldate'].dt.year)
    .groupby('received_year')[['crash', 'fire']]
    .agg(lambda s: round(float((s == 'N').mean() * 100), 2))
    .reset_index()
)

print("Percent of rows equal to zero by received year")
display(zero_watch)

print("Percent of rows equal to N by received year")
display(yn_watch)

## 6. Focused Review Of Open Issues

This section is where we pressure test the flags that still need judgment.

Instead of blindly cleaning them, we inspect the actual patterns first and decide what is safe to trust.

In [ ]:
vehicle_mask = std_df['prod_type'].eq('V').fillna(False)
vehicle_std = std_df.loc[vehicle_mask].copy()
vehicle_flags = flag_df.loc[vehicle_mask].copy()
vehicle_std['lag_days'] = (vehicle_std['ldate'] - vehicle_std['faildate']).dt.days

open_review = pd.DataFrame([
    {
        'topic': 'negative incident-to-receipt lag',
        'rows': int(vehicle_std['lag_days'].lt(0).sum()),
        'row_pct': round(float(vehicle_std['lag_days'].lt(0).sum() / len(vehicle_std) * 100), 4),
        'current_read': 'rare and likely administrative'
    },
    {
        'topic': 'fail year < 1990 for a 1990+ vehicle',
        'rows': int(vehicle_flags['flag_fail_old_new_vehicle'].sum()),
        'row_pct': round(float(vehicle_flags['flag_fail_old_new_vehicle'].sum() / len(vehicle_std) * 100), 4),
        'current_read': 'looks like date entry noise rather than a true incident year'
    },
    {
        'topic': 'fail year before model_year - 1',
        'rows': int(vehicle_flags['flag_fail_pre_model'].sum()),
        'row_pct': round(float(vehicle_flags['flag_fail_pre_model'].sum() / len(vehicle_std) * 100), 4),
        'current_read': 'worth flagging before any date-derived features are built'
    },
    {
        'topic': 'incident-to-receipt lag above 365 days',
        'rows': int(vehicle_std['lag_days'].gt(365).sum()),
        'row_pct': round(float(vehicle_std['lag_days'].gt(365).sum() / len(vehicle_std) * 100), 4),
        'current_read': 'often looks like real reporting delay rather than broken chronology'
    },
    {
        'topic': 'veh_speed == 999',
        'rows': int(vehicle_flags['flag_speed_999'].sum()),
        'row_pct': round(float(vehicle_flags['flag_speed_999'].sum() / len(vehicle_std) * 100), 4),
        'current_read': 'clear sentinel candidate'
    },
    {
        'topic': 'veh_speed > 200',
        'rows': int(vehicle_flags['flag_speed_high'].sum()),
        'row_pct': round(float(vehicle_flags['flag_speed_high'].sum() / len(vehicle_std) * 100), 4),
        'current_read': 'rare but worth hand review before nulling'
    },
    {
        'topic': 'miles > 500000',
        'rows': int(vehicle_flags['flag_miles_high'].sum()),
        'row_pct': round(float(vehicle_flags['flag_miles_high'].sum() / len(vehicle_std) * 100), 4),
        'current_read': 'rare and possibly real for high-use vehicles'
    },
    {
        'topic': 'vin length != 11',
        'rows': int(vehicle_flags['flag_vin_len_bad'].sum()),
        'row_pct': round(float(vehicle_flags['flag_vin_len_bad'].sum() / len(vehicle_std) * 100), 4),
        'current_read': 'quality flag, not an automatic row drop'
    }
])

lag_non_null = vehicle_std['lag_days'].dropna()
lag_profile = pd.DataFrame([
    {
        'metric': 'rows with lag available',
        'value': int(lag_non_null.shape[0])
    },
    {
        'metric': 'lag median days',
        'value': int(lag_non_null.median())
    },
    {
        'metric': 'lag 95th percentile days',
        'value': int(lag_non_null.quantile(0.95))
    },
    {
        'metric': 'lag 99th percentile days',
        'value': int(lag_non_null.quantile(0.99))
    }
])

print("Vehicle-only open-issue summary:")
display(open_review)

print("\nLag profile from incident date to complaint receipt:")
display(lag_profile)

### 6.1 Date Lag Review

We look at the lag between incident date and complaint receipt date.

The goal is to separate harmless quirks from dates that should not be used for derived time features.

In [ ]:
lag_cols = [
    'cmplid',
    'odino',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'faildate',
    'ldate',
    'datea',
    'lag_days',
    'cdescr'
]

lag_non_null = vehicle_std['lag_days'].dropna()
lag_bands = pd.DataFrame([
    {'band': '< 0 days', 'rows': int(lag_non_null.lt(0).sum())},
    {'band': '0 days', 'rows': int(lag_non_null.eq(0).sum())},
    {'band': '1 day', 'rows': int(lag_non_null.eq(1).sum())},
    {'band': '2 to 7 days', 'rows': int(lag_non_null.between(2, 7).sum())},
    {'band': '8 to 30 days', 'rows': int(lag_non_null.between(8, 30).sum())},
    {'band': '31 to 365 days', 'rows': int(lag_non_null.between(31, 365).sum())},
    {'band': '366 to 1000 days', 'rows': int(lag_non_null.between(366, 1000).sum())},
    {'band': '1001+ days','rows': int(lag_non_null.gt(1000).sum())}
])
lag_bands['row_pct'] = (lag_bands['rows'] / len(vehicle_std) * 100).round(4)

negative_lag = vehicle_std.loc[vehicle_std['lag_days'].lt(0), lag_cols].sort_values(['lag_days', 'ldate', 'faildate'])
long_lag = vehicle_std.loc[vehicle_std['lag_days'].gt(1000), lag_cols].sort_values('lag_days', ascending=False).head(10)

print("Vehicle incident-to-receipt lag bands:")
display(lag_bands)

print("\nNegative-lag rows to inspect by hand:")
display(negative_lag)

print("\nVery long-lag rows to inspect for modeling context:")
display(long_lag)

lag_plot = lag_non_null.loc[lag_non_null.between(0, 365)]
plt.figure(figsize=(11, 4))
sns.histplot(lag_plot, bins=60)
plt.title("Vehicle Complaint Lag From Incident To Receipt (0-365 Days)")
plt.xlabel("Days From Incident To Receipt")
plt.ylabel("Complaint Rows")
plt.tight_layout()
plt.show()

### 6.2 Date Field Sanity Beyond Lag

Some date problems are not simple lag issues.

This looks for implausible incident years and failures that do not line up with vehicle model year.

In [ ]:
date_cols = [
    'cmplid',
    'odino',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'faildate',
    'ldate',
    'datea',
    'lag_days',
    'cdescr'
]

vehicle_std['fail_year'] = vehicle_std['faildate'].dt.year
vehicle_std['ldate_year'] = vehicle_std['ldate'].dt.year

fail_sanity = pd.DataFrame([
    {
        'check': 'fail year < 1990 for a 1990+ vehicle',
        'rows': int(vehicle_flags['flag_fail_old_new_vehicle'].sum()),
        'row_pct': round(float(vehicle_flags['flag_fail_old_new_vehicle'].sum() / len(vehicle_std) * 100), 4),
        'current_read': 'very likely data entry noise'
    },
    {
        'check': 'fail year before model_year - 1',
        'rows': int(vehicle_flags['flag_fail_pre_model'].sum()),
        'row_pct': round(float(vehicle_flags['flag_fail_pre_model'].sum() / len(vehicle_std) * 100), 4),
        'current_read': 'safe to flag before any time-derived modeling'
    },
    {
        'check': 'fail year before model_year - 5',
        'rows': int(vehicle_flags['flag_fail_pre_model_far'].sum()),
        'row_pct': round(float(vehicle_flags['flag_fail_pre_model_far'].sum() / len(vehicle_std) * 100), 4),
        'current_read': 'looks too inconsistent to treat as a real incident year'
    },
    {
        'check': 'fail year < 1980',
        'rows': int(vehicle_std['fail_year'].lt(1980).sum()),
        'row_pct': round(float(vehicle_std['fail_year'].lt(1980).sum() / len(vehicle_std) * 100), 4),
        'current_read': 'mixture of likely bad years and a few legacy vehicles'
    }
])

odd_fail_years = (
    vehicle_std.loc[vehicle_flags['flag_fail_pre_model'], 'fail_year']
    .value_counts()
    .head(15)
    .rename_axis('fail_year')
    .reset_index(name='rows')
)
old_fail_rows = vehicle_std.loc[vehicle_flags['flag_fail_old_new_vehicle'], date_cols].sort_values(['faildate', 'ldate']).head(12)
pre_model_rows = vehicle_std.loc[vehicle_flags['flag_fail_pre_model'], date_cols].sort_values(['faildate', 'ldate']).head(12)

print("Date sanity checks for vehicle complaints:")
display(fail_sanity)

print("\nMost common fail years among pre-model incidents:")
display(odd_fail_years)

print("\nRows with very old fail years on modern vehicles:")
display(old_fail_rows)

print("\nRows where fail year is before model year minus one:")
display(pre_model_rows)

plt.figure(figsize=(10, 5))
sns.barplot(data=odd_fail_years, x='rows', y='fail_year', orient='h')
plt.title("Most Common Suspicious Fail Years In The Vehicle Cohort")
plt.xlabel("Complaint Rows")
plt.ylabel("Fail Year")
plt.tight_layout()
plt.show()

### 6.3 High Speed Review

Here we inspect unusually high reported speeds or values of 999 for unknown.

This helps decide whether extreme values should stay as raw values, become missing, or just be flagged.

In [ ]:
speed_cols = [
    'cmplid',
    'odino',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'veh_speed',
    'miles',
    'faildate',
    'cdescr'
]

speed_thresholds = pd.DataFrame([
    {'threshold': '> 120', 'rows': int(vehicle_std['veh_speed'].gt(120).sum())},
    {'threshold': '> 160', 'rows': int(vehicle_std['veh_speed'].gt(160).sum())},
    {'threshold': '> 200', 'rows': int(vehicle_std['veh_speed'].gt(200).sum())},
    {'threshold': '== 999', 'rows': int(vehicle_std['veh_speed'].eq(999).sum())}
])
speed_thresholds['row_pct'] = (speed_thresholds['rows'] / len(vehicle_std) * 100).round(4)

high_speed_vals = (
    vehicle_std.loc[vehicle_std['veh_speed'].gt(200), 'veh_speed']
    .value_counts()
    .head(15)
    .rename_axis('veh_speed')
    .reset_index(name='rows')
)

high_speed_makes = (
    vehicle_std.loc[vehicle_std['veh_speed'].gt(200), 'maketxt']
    .value_counts()
    .head(10)
    .rename_axis('maketxt')
    .reset_index(name='rows')
)

high_speed_rows = vehicle_std.loc[vehicle_std['veh_speed'].gt(200), speed_cols].sort_values('veh_speed', ascending=False).head(12)

print("Vehicle speed thresholds:")
display(speed_thresholds)

print("\nMost common values above 200:")
display(high_speed_vals)

print("\nMakes most represented among speeds above 200:")
display(high_speed_makes)

print("\nRows above 200 to inspect by hand:")
display(high_speed_rows)

plot_speed = vehicle_std.loc[vehicle_std['veh_speed'].between(0, 120), 'veh_speed'].dropna()
plt.figure(figsize=(11, 4))
sns.histplot(plot_speed, bins=50)
plt.title("Vehicle Speed Distribution For Plausible Values (0-120)")
plt.xlabel("Vehicle Speed")
plt.ylabel("Complaint Rows")
plt.tight_layout()
plt.show()

### 6.4 High Mileage Review

Mileage is trickier than speed because very high values can be real for heavily used vehicles.

We review the pattern before deciding whether to keep, flag, cap, or null anything later.

In [ ]:
miles_cols = [
    'cmplid',
    'odino',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'miles',
    'veh_speed',
    'faildate',
    'cdescr'
]

miles_thresholds = pd.DataFrame([
    {'threshold': '> 200000', 'rows': int(vehicle_std['miles'].gt(200000).sum())},
    {'threshold': '> 300000', 'rows': int(vehicle_std['miles'].gt(300000).sum())},
    {'threshold': '> 500000', 'rows': int(vehicle_std['miles'].gt(500000).sum())},
    {'threshold': '> 1000000', 'rows': int(vehicle_std['miles'].gt(1000000).sum())}
])
miles_thresholds['row_pct'] = (miles_thresholds['rows'] / len(vehicle_std) * 100).round(4)

high_miles_vals = (
    vehicle_std.loc[vehicle_std['miles'].gt(500000), 'miles']
    .value_counts()
    .head(15)
    .rename_axis('miles')
    .reset_index(name='rows')
)

high_miles_makes = (
    vehicle_std.loc[vehicle_std['miles'].gt(500000), 'maketxt']
    .value_counts()
    .head(10)
    .rename_axis('maketxt')
    .reset_index(name='rows')
)

high_miles_rows = vehicle_std.loc[vehicle_std['miles'].gt(500000), miles_cols].sort_values('miles', ascending=False).head(12)

print("Vehicle mileage thresholds:")
display(miles_thresholds)

print("\nMost common values above 500000 miles:")
display(high_miles_vals)

print("\nMakes most represented among mileages above 500000:")
display(high_miles_makes)

print("\nRows above 500000 miles to inspect by hand:")
display(high_miles_rows)

plot_miles = vehicle_std.loc[vehicle_std['miles'].between(0, 300000), 'miles'].dropna()
plt.figure(figsize=(11, 4))
sns.histplot(plot_miles, bins=60)
plt.title("Vehicle Mileage At Failure For Plausible Values (0-300k)")
plt.xlabel("Mileage At Failure")
plt.ylabel("Complaint Rows")
plt.tight_layout()
plt.show()

### 6.5 Zero Review By Field

Zeros are not automatically harmless in this dataset because of the documented file changes.

This section checks whether zero looks like a true observed value or something more ambiguous by field.

In [ ]:
zero_cols = [
    'cmplid',
    'odino',
    'maketxt',
    'modeltxt',
    'ldate',
    'miles',
    'veh_speed',
    'injured',
    'deaths',
    'cdescr'
]

zero_fields = [
    'injured',
    'deaths',
    'miles',
    'veh_speed'
]

zero_read = {
    'injured': 'keep zero for now because positive injury counts appear consistently every year',
    'deaths': 'keep zero for now because the field behaves like a sparse count rather than a placeholder bucket',
    'miles': 'keep zero but treat it as ambiguous and worth a dedicated flag later',
    'veh_speed': 'keep zero but treat it as context-dependent because many complaints happen while parked or starting up'
}

zero_summary_rows = []
for col in zero_fields:
    non_null = int(vehicle_std[col].notna().sum())
    zero_rows = int(vehicle_std[col].eq(0).sum())
    positive_rows = int(vehicle_std[col].gt(0).sum())
    zero_summary_rows.append({
        'field': col,
        'non_null_rows': non_null,
        'zero_rows': zero_rows,
        'zero_pct_of_non_null': round(float(zero_rows / non_null * 100), 2),
        'positive_rows': positive_rows,
        'positive_pct_of_non_null': round(float(positive_rows / non_null * 100), 2),
        'current_read': zero_read[col]
    })
zero_summary = pd.DataFrame(zero_summary_rows)

period_rows = []
for period_name, mask in [
    ('2020-2021', vehicle_std['ldate_year'].le(2021)),
    ('2022-2026', vehicle_std['ldate_year'].ge(2022))
]:
    part = vehicle_std.loc[mask]
    for col in zero_fields:
        non_null = int(part[col].notna().sum())
        zero_rows = int(part[col].eq(0).sum())
        positive_rows = int(part[col].gt(0).sum())
        period_rows.append({
            'field': col,
            'period': period_name,
            'non_null_rows': non_null,
            'zero_pct_of_non_null': round(float(zero_rows / non_null * 100), 2),
            'positive_pct_of_non_null': round(float(positive_rows / non_null * 100), 2)
        })
zero_period = pd.DataFrame(period_rows)

injury_examples = vehicle_std.loc[vehicle_std['injured'].gt(0), zero_cols].head(8)
death_examples = vehicle_std.loc[vehicle_std['deaths'].gt(0), zero_cols].head(8)
zero_miles_examples = vehicle_std.loc[vehicle_std['miles'].eq(0), zero_cols].head(10)
zero_speed_examples = vehicle_std.loc[vehicle_std['veh_speed'].eq(0), zero_cols].head(10)

print("Zero review by field:")
display(zero_summary)

print("\nZero and positive rates before and after the 2021 file change window:")
display(zero_period)

print("\nExamples where injured > 0:")
display(injury_examples)

print("\nExamples where deaths > 0:")
display(death_examples)

print("\nExamples where miles == 0:")
display(zero_miles_examples)

print("\nExamples where veh_speed == 0:")
display(zero_speed_examples)

plt.figure(figsize=(10, 5))
sns.barplot(data=zero_period, x='zero_pct_of_non_null', y='field', hue='period')
plt.title("Zero Rate By Field Before And After The 2021 File Change")
plt.xlabel("Zero Share Of Non-Null Rows")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 6.6 Current Direction After Review

After the focused checks, we summarize the current thoughts on what is safe to do now and what should stay as a flagged open question.

These decisions will be used to choose what candidate features should be confidently added to the first cleaning layer.

In [ ]:
review_takeaways = pd.DataFrame([
    {
        'topic': 'date anchor',
        'base_clean_lean': 'keep raw dates',
        'model_ready_lean': 'use ldate as the anchor for all first-pass model tables',
        'why': 'receipt date is the cleanest operational time point in the current extract'
    },
    {
        'topic': 'faildate trust rule',
        'base_clean_lean': 'keep raw faildate but add trust and untrusted flags',
        'model_ready_lean': "derive lag only when faildate <= ldate and fail_year >= model_year - 1",
        'why': 'this keeps useful incident timing while blocking the obviously broken date cases'
    },
    {
        'topic': 'miles and veh_speed zeros',
        'base_clean_lean': 'keep zeros as raw values',
        'model_ready_lean': 'add missingness flags but not zero flags in v1',
        'why': 'zero can be semantically mixed, but missingness is the stronger first-pass signal'
    },
    {
        'topic': 'severity label',
        'base_clean_lean': 'create row-level primary and broad severity flags',
        'model_ready_lean': "collapse the primary flag to an odino-level case table for severity ranking",
        'why': 'severity ranking should prioritize complaints, not overweight complaints that were split into more component rows'
    },
    {
        'topic': 'severity baseline features',
        'base_clean_lean': 'leave target-defining severity columns in the base layer',
        'model_ready_lean': 'exclude injured, deaths, fire, crash, medical_attn, vehicles_towed_yn, and datea from the first baseline',
        'why': 'this keeps the severity model leakage-safe'
    },
    {
        'topic': 'component model table',
        'base_clean_lean': 'keep the vehicle complaint-component rows',
        'model_ready_lean': 'use compdesc as the raw target source until cleaned component groups are finalized',
        'why': 'the component task naturally lives on the complaint-component row rather than the collapsed case row'
    }
])

display(review_takeaways)

## 7. High Confidence Candidate Decisions

Only the cleaning rules that now look clearly justified get applied here.

Want to build a candidate cleaned table without pretending every open issue is settled.

In [ ]:
candidate_df = std_df.copy()

candidate_df.loc[flag_df['flag_prod_type_bad'], 'prod_type'] = pd.NA
candidate_df.loc[flag_df['flag_year_unknown'] | flag_df['flag_year_out_of_range'], 'yeartxt'] = pd.NA
candidate_df.loc[flag_df['flag_speed_999'], 'veh_speed'] = pd.NA
candidate_df.loc[flag_df['flag_injured_99'], 'injured'] = pd.NA
candidate_df.loc[flag_df['flag_deaths_99'], 'deaths'] = pd.NA
candidate_df.loc[flag_df['flag_state_bad'], 'state'] = pd.NA
candidate_df.loc[flag_df['flag_dealer_state_bad'], 'dealer_state'] = pd.NA

known_model_year = candidate_df['yeartxt'].notna()
fail_year = candidate_df['faildate'].dt.year
trusted_faildate = (
    candidate_df['faildate'].notna()
    & candidate_df['ldate'].notna()
    & candidate_df['faildate'].le(candidate_df['ldate'])
    & (~known_model_year | fail_year.ge(candidate_df['yeartxt'] - 1))
)

candidate_df['miles_missing_flag'] = candidate_df['miles'].isna().astype('Int8')
candidate_df['veh_speed_missing_flag'] = candidate_df['veh_speed'].isna().astype('Int8')
candidate_df['faildate_trusted_flag'] = trusted_faildate.astype('Int8')
candidate_df['faildate_untrusted_flag'] = (candidate_df['faildate'].notna() & ~trusted_faildate).astype('Int8')
candidate_df['lag_days_safe'] = pd.Series(pd.NA, index=candidate_df.index, dtype='Int64')
candidate_df.loc[trusted_faildate, 'lag_days_safe'] = (
    candidate_df.loc[trusted_faildate, 'ldate'] - candidate_df.loc[trusted_faildate, 'faildate']
).dt.days.astype('Int64')

candidate_df['severity_primary_row_flag'] = (
    candidate_df['deaths'].fillna(0).gt(0)
    | candidate_df['injured'].fillna(0).gt(0)
    | candidate_df['fire'].eq('Y').fillna(False)
    | candidate_df['crash'].eq('Y').fillna(False)
).astype('Int8')

candidate_df['severity_broad_row_flag'] = (
    candidate_df['severity_primary_row_flag'].eq(1)
    | candidate_df['medical_attn'].eq('Y').fillna(False)
    | candidate_df['vehicles_towed_yn'].eq('Y').fillna(False)
).astype('Int8')

candidate_rules = pd.DataFrame([
    {
        'field_or_feature': 'prod_type',
        'rule': 'invalid product type to missing',
        'affected_rows': int(flag_df['flag_prod_type_bad'].fillna(False).sum()),
        'why': 'schema-coded field with a small and explicit valid set'
    },
    {
        'field_or_feature': 'yeartxt',
        'rule': '9999 and out-of-range values to missing',
        'affected_rows': int((flag_df['flag_year_unknown'] | flag_df['flag_year_out_of_range']).fillna(False).sum()),
        'why': 'CMPL.txt documents 9999 as unknown and extreme years are not usable model years'
    },
    {
        'field_or_feature': 'veh_speed',
        'rule': '999 sentinel to missing',
        'affected_rows': int(flag_df['flag_speed_999'].fillna(False).sum()),
        'why': '999 behaves like a placeholder value'
    },
    {
        'field_or_feature': 'injured / deaths',
        'rule': '99 sentinel to missing',
        'affected_rows': int(flag_df['flag_injured_99'].fillna(False).sum() + flag_df['flag_deaths_99'].fillna(False).sum()),
        'why': '99 is not a plausible person-count code here'
    },
    {
        'field_or_feature': 'state / dealer_state',
        'rule': 'invalid codes to missing',
        'affected_rows': int(flag_df['flag_state_bad'].fillna(False).sum() + flag_df['flag_dealer_state_bad'].fillna(False).sum()),
        'why': 'invalid geography codes are not analytically useful as location features'
    },
    {
        'field_or_feature': 'lag_days_safe',
        'rule': "derive only when faildate <= ldate and fail_year >= model_year - 1",
        'affected_rows': int(trusted_faildate.sum()),
        'why': 'keeps incident-date features while blocking clearly broken or incompatible dates'
    },
    {
        'field_or_feature': 'miles_missing_flag / veh_speed_missing_flag',
        'rule': 'add missingness indicators for first structured baselines',
        'affected_rows': int(candidate_df['miles_missing_flag'].sum() + candidate_df['veh_speed_missing_flag'].sum()),
        'why': 'missingness is useful while zero stays a raw value rather than a new flag'
    },
    {
        'field_or_feature': 'severity_primary_row_flag',
        'rule': "1 if deaths > 0 or injured > 0 or fire == 'Y' or crash == 'Y'",
        'affected_rows': int(candidate_df['severity_primary_row_flag'].sum()),
        'why': 'this is the first-pass severity label that will later be collapsed to odino level'
    },
    {
        'field_or_feature': 'severity_broad_row_flag',
        'rule': "primary label plus medical_attn == 'Y' or vehicles_towed_yn == 'Y'",
        'affected_rows': int(candidate_df['severity_broad_row_flag'].sum()),
        'why': 'use this only as a sensitivity label, not the main target'
    }
])

candidate_features = pd.DataFrame([
    {
        'feature': 'faildate_trusted_flag',
        'row_count': int(candidate_df['faildate_trusted_flag'].sum())
    },
    {
        'feature': 'faildate_untrusted_flag',
        'row_count': int(candidate_df['faildate_untrusted_flag'].sum())
    },
    {
        'feature': 'lag_days_safe non-null',
        'row_count': int(candidate_df['lag_days_safe'].notna().sum())
    },
    {
        'feature': 'miles_missing_flag',
        'row_count': int(candidate_df['miles_missing_flag'].sum())
    },
    {
        'feature': 'veh_speed_missing_flag',
        'row_count': int(candidate_df['veh_speed_missing_flag'].sum())
    },
    {
        'feature': 'severity_primary_row_flag',
        'row_count': int(candidate_df['severity_primary_row_flag'].sum())
    },
    {
        'feature': 'severity_broad_row_flag',
        'row_count': int(candidate_df['severity_broad_row_flag'].sum())
    }
])

print("Locked candidate rules:")
display(candidate_rules)

print()
print("Key model-ready features from this pass:")
display(candidate_features)

### 7.1 Impact Summary

We compare the candidate cleaning against the earlier layer to see what actually changed.

Keeps the notebook grounded in measurable impact instead of intuition.

In [ ]:
compare_cols = [
    'yeartxt',
    'miles',
    'veh_speed',
    'injured',
    'deaths',
    'purch_dt',
    'manuf_dt',
    'state',
    'dealer_state',
    'maketxt',
    'modeltxt',
    'city',
    'dealer_city'
]

null_compare = pd.DataFrame({
    'raw_null_pct': (raw_df[compare_cols].isna().mean() * 100).round(2),
    'std_null_pct': (std_df[compare_cols].isna().mean() * 100).round(2),
    'candidate_null_pct': (candidate_df[compare_cols].isna().mean() * 100).round(2)
}).reset_index().rename(columns={'index': 'column'})

dtype_compare = pd.DataFrame({
    'raw_dtype': raw_df[compare_cols].dtypes.astype('string'),
    'std_dtype': std_df[compare_cols].dtypes.astype('string'),
    'candidate_dtype': candidate_df[compare_cols].dtypes.astype('string')
}).reset_index().rename(columns={'index': 'column'})

checks = pd.DataFrame([
        {'metric': 'rows after candidate cleaning', 'value': len(candidate_df)},
        {'metric': 'columns in candidate frame', 'value': candidate_df.shape[1]},
        {'metric': 'cmplid unique after candidate cleaning', 'value': candidate_df['cmplid'].nunique(dropna=True)},
        {'metric': 'odino unique after candidate cleaning', 'value': candidate_df['odino'].nunique(dropna=True)},
        {'metric': 'full-row duplicates after candidate cleaning', 'value': int(candidate_df.duplicated().sum())}
])

print("Null percentages by layer: ")
display(null_compare.sort_values('candidate_null_pct', ascending=False))

print("\nData type by layer: ")
display(dtype_compare)

print("\nGeneral checks after cleaning: ")
display(checks)

plot_df = null_compare.melt(id_vars='column', var_name='stage', value_name='null_pct')

plt.figure(figsize=(12, 8))
sns.barplot(data=plot_df, x='null_pct', y='column', hue='stage')
plt.title("Null Percentage by Cleaning Stage")
plt.xlabel("Null Percentage")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 7.2 Raw Versus Candidate Review

We inspect flagged values side by side so we can see exactly what the candidate rules changed.

That is the final sanity check before using the cleaned layer for modeling previews.

In [ ]:
review_cols = [
    'cmplid',
    'odino',
    'prod_type',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'state',
    'dealer_state',
    'vin',
    'faildate',
    'datea',
    'ldate',
    'miles',
    'veh_speed',
    'injured',
    'deaths',
    'cdescr'
]

print("Chronology rows to review:")
display(compare_view(flag_df['flag_date_order_bad'], review_cols, n=10))

print("\nBad state rows to review:")
display(compare_view(flag_df['flag_state_bad'] | flag_df['flag_dealer_state_bad'], review_cols, n=10))

print("\nModel year rows to review:")
display(compare_view(flag_df['flag_year_unknown'] | flag_df['flag_year_out_of_range'], review_cols, n=10))

print("\nSpeed and miles rows to review:")
display(compare_view(flag_df['flag_speed_999'] | flag_df['flag_speed_high'] | flag_df['flag_miles_high'], review_cols, n=10))

print("\nVIN rows to review:")
display(compare_view(flag_df['flag_vin_len_bad'], review_cols, n=10))

## 8. Vehicle-Ready Outputs

With the candidate cleaning in place, we preview the vehicle focused layer that will be used for the first structured modeling work.

This confirms the cleaned fields still behave sensibly once we restrict to the modeling cohort to vehicles.

In [ ]:
vehicle_mask = candidate_df['prod_type'].eq('V').fillna(False)
vehicle_candidate = candidate_df.loc[vehicle_mask].copy()

vehicle_ready = pd.DataFrame([
    {'metric': 'vehicle rows', 'value': len(vehicle_candidate)},
    {'metric': 'vehicle row share pct', 'value': round(float(len(vehicle_candidate) / len(candidate_df) * 100), 2)},
    {'metric': 'vehicle unique makes', 'value': vehicle_candidate['maketxt'].nunique(dropna=True)},
    {'metric': 'vehicle unique models', 'value': vehicle_candidate['modeltxt'].nunique(dropna=True)}
])

vehicle_core = pd.DataFrame({
    'vehicle_non_null_pct': (vehicle_candidate[[
            'maketxt',
            'modeltxt',
            'yeartxt',
            'state',
            'miles',
            'veh_speed',
            'injured',
            'deaths',
            'cdescr'
        ]].notna().mean() * 100
    ).round(2)
}).reset_index().rename(columns={'index': 'column'})

vehicle_flags = pd.DataFrame([
    {
        'flag': col,
        'vehicle_row_pct': round(float(flag_df.loc[vehicle_candidate.index, col].fillna(False).sum() / len(vehicle_candidate) * 100), 4)
    }
    for col in flag_df.columns
]).sort_values('vehicle_row_pct', ascending=False)

print("General vehicle cohort overview:")
display(vehicle_ready)

print("\nCore vehicle features coverage:")
display(vehicle_core)

print("\nPercent of vehicle cohort rows flagged bad or suspicious:")
display(vehicle_flags)

### 8.1 Structured Model Table Preview

We map the cleaned candidate DataFrame into the first structured modeling views.

This is where we start translating specific cleaning decisions into the tables that will be used for each project task/model.

In [ ]:
severity_policy = pd.DataFrame([
    {
        'decision': 'unit of analysis',
        'value': 'one complaint case per odino'
    },
    {
        'decision': 'primary severity label',
        'value': "deaths > 0 or injured > 0 or fire == 'Y' or crash == 'Y'"
    },
    {
        'decision': 'broader sensitivity label',
        'value': "primary label plus medical_attn == 'Y' or vehicles_towed_yn == 'Y'"
    },
    {
        'decision': 'excluded from first severity baseline',
        'value': 'injured, deaths, fire, crash, medical_attn, vehicles_towed_yn, datea'
    }
])

component_policy = pd.DataFrame([
    {
        'decision': 'unit of analysis',
        'value': 'vehicle complaint-component row'
    },
    {
        'decision': 'raw target source',
        'value': 'compdesc'
    },
    {
        'decision': 'first target cleanup step',
        'value': 'standardize raw component labels and then build logical component groups'
    },
    {
        'decision': 'first baseline target form',
        'value': 'cleaned component group after grouping work is finished'
    }
])

multi_case_counts = vehicle_candidate.groupby('odino').size()
multi_case_odino = multi_case_counts[multi_case_counts > 1].index
multi_case_df = vehicle_candidate.loc[vehicle_candidate['odino'].isin(multi_case_odino)].copy()
case_conflict = pd.DataFrame([
    {
        'column': col,
        'multi_case_conflicts': int(multi_case_df.groupby('odino')[col].nunique(dropna=True).gt(1).sum())
    }
    for col in [
        'maketxt',
        'modeltxt',
        'yeartxt',
        'state',
        'faildate',
        'ldate',
        'datea',
        'miles',
        'veh_speed',
        'injured',
        'deaths',
        'fire',
        'crash'
    ]
]).sort_values(['multi_case_conflicts', 'column'], ascending=[False, True])

severity_case_df = (
    vehicle_candidate.sort_values(['odino', 'cmplid'])
    .groupby('odino', dropna=False)
    .agg(
        row_count=('cmplid', 'size'),
        component_count=('compdesc', lambda s: s.astype('string').dropna().nunique()),
        component_list=('compdesc', lambda s: ' | '.join(sorted(set(s.dropna().astype(str)))[:4])),
        maketxt=('maketxt', 'first'),
        modeltxt=('modeltxt', 'first'),
        yeartxt=('yeartxt', 'first'),
        state=('state', 'first'),
        ldate=('ldate', 'first'),
        faildate=('faildate', 'first'),
        lag_days_safe=('lag_days_safe', 'first'),
        miles=('miles', 'first'),
        veh_speed=('veh_speed', 'first'),
        miles_missing_flag=('miles_missing_flag', 'max'),
        veh_speed_missing_flag=('veh_speed_missing_flag', 'max'),
        faildate_untrusted_flag=('faildate_untrusted_flag', 'max'),
        severity_primary_flag=('severity_primary_row_flag', 'max'),
        severity_broad_flag=('severity_broad_row_flag', 'max')
    ).reset_index()
)

component_model_df = vehicle_candidate.loc[vehicle_candidate['compdesc'].notna(), [
    'cmplid',
    'odino',
    'ldate',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'state',
    'miles',
    'veh_speed',
    'miles_missing_flag',
    'veh_speed_missing_flag',
    'faildate_untrusted_flag',
    'lag_days_safe',
    'compdesc'
]].rename(columns={'compdesc': 'component_raw'}).copy()

severity_summary = pd.DataFrame([
    {
        'metric': 'severity case rows',
        'value': len(severity_case_df)
    },
    {
        'metric': 'cases with >1 component row',
        'value': int((severity_case_df['row_count'] > 1).sum())
    },
    {
        'metric': 'primary high-risk case rate pct',
        'value': round(float(severity_case_df['severity_primary_flag'].mean() * 100), 2)
    },
    {
        'metric': 'broad high-risk case rate pct',
        'value': round(float(severity_case_df['severity_broad_flag'].mean() * 100), 2)
    },
    {
        'metric': 'cases with untrusted faildate pct',
        'value': round(float(severity_case_df['faildate_untrusted_flag'].mean() * 100), 2)
    }
])

component_summary = pd.DataFrame([
    {
        'metric': 'component-model rows',
        'value': len(component_model_df)
    },
    {
        'metric': 'unique complaint cases in component table',
        'value': component_model_df['odino'].nunique(dropna=True)
    },
    {
        'metric': 'unique raw component labels',
        'value': component_model_df['component_raw'].nunique(dropna=True)
    },
    {
        'metric': 'rows with untrusted faildate pct',
        'value': round(float(component_model_df['faildate_untrusted_flag'].mean() * 100), 2)
    }
])

component_top = (
    component_model_df['component_raw']
    .value_counts()
    .head(15)
    .rename_axis('component_raw')
    .reset_index(name='rows')
)

severity_preview = severity_case_df.loc[:,[
    'odino',
    'row_count',
    'component_count',
    'component_list',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'ldate',
    'lag_days_safe',
    'faildate_untrusted_flag',
    'severity_primary_flag',
    'severity_broad_flag'
]].head(10)

print("Severity model policy:")
display(severity_policy)

print("\nComponent model policy:")
display(component_policy)

print("\nODINO collapse consistency check for multi-row cases:")
display(case_conflict)

print("\nSeverity case-level table summary:")
display(severity_summary)

print("\nSample severity case rows:")
display(severity_preview)

print("\nComponent row-level table summary:")
display(component_summary)

print("\nTop raw component labels before grouping:")
display(component_top)

## 9. Component Target Design

The component task needs a cleaned and grouped target, not the raw component strings as is.

This builds and checks the grouped component design before it moves into the pipeline.

In [ ]:
comp_target = component_model_df.copy()

comp_target['component_raw_std'] = (
    comp_target['component_raw']
    .astype('string')
    .str.upper()
    .str.replace(r'\s*;\s*', ':', regex=True)
    .str.replace(r'\s*:\s*', ':', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

comp_target['component_parent'] = (
    comp_target['component_raw_std']
    .str.split(':')
    .str[0]
    .str.strip()
)

drop_child_parents = {
    'CHEST CLIP, BUCKLE, HARNESS',
    'CARRY HANDLE, SHELL, BASE',
    'CHILD SEAT',
    'TETHER, LOWER ANCHOR (ON CAR SEAT OR VEHICLE)',
    'I SUSPECT THE CAR SEAT IS COUNTERFEIT',
    'INSERT, PADDING'
}


def map_component_group(label):
    parent = label.split(':', 1)[0].strip()

    if label in {'UNKNOWN OR OTHER', 'OTHER/I AM NOT SURE'}:
        return 'UNKNOWN OR OTHER'

    if parent in {'COMMUNICATION', 'COMMUNICATIONS'}:
        return 'COMMUNICATION'

    if parent in {'ENGINE', 'ENGINE AND ENGINE COOLING'}:
        return 'ENGINE / COOLING'

    if parent in {'VISIBILITY', 'VISIBILITY/WIPER'}:
        return 'VISIBILITY / WIPER'

    if parent in {
        'FUEL/PROPULSION SYSTEM',
        'FUEL SYSTEM, GASOLINE',
        'FUEL SYSTEM, DIESEL',
        'FUEL SYSTEM, OTHER',
        'HYBRID PROPULSION SYSTEM'
    }:
        return 'FUEL / PROPULSION'

    if parent in {
        'SERVICE BRAKES',
        'SERVICE BRAKES, HYDRAULIC',
        'SERVICE BRAKES, AIR',
        'SERVICE BRAKES, ELECTRIC'
    }:
        return 'SERVICE BRAKES'

    return parent


comp_target['component_group'] = comp_target['component_raw_std'].map(map_component_group)

comp_target['drop_reason'] = pd.NA

comp_target.loc[
    comp_target['component_parent'].isin(drop_child_parents),
    'drop_reason'
] = 'DROP_CHILD_RESTRAINT'

comp_target.loc[
    comp_target['component_group'].isin({'EQUIPMENT', 'EQUIPMENT ADAPTIVE/MOBILITY'}),
    'drop_reason'
] = 'DROP_EQUIPMENT'

comp_target.loc[
    comp_target['component_group'].eq('UNKNOWN OR OTHER'),
    'drop_reason'
] = 'DROP_UNKNOWN_OTHER'

group_counts = (
    comp_target.loc[comp_target['drop_reason'].isna(), 'component_group']
    .value_counts()
)

comp_target['component_group_rows'] = comp_target['component_group'].map(group_counts)

comp_target.loc[
    comp_target['drop_reason'].isna() & comp_target['component_group_rows'].lt(250),
    'drop_reason'
] = 'DROP_RARE_GROUP'

comp_target['keep_for_component_model'] = comp_target['drop_reason'].isna()

component_map_final = (
    comp_target.groupby(
        [
            'component_raw_std',
            'component_parent',
            'component_group',
            'component_group_rows',
            'drop_reason',
            'keep_for_component_model'
        ],
        dropna=False
    )
    .size()
    .reset_index(name='rows')
    .sort_values(
        ['keep_for_component_model', 'rows', 'component_group', 'component_raw_std'],
        ascending=[False, False, True, True]
    )
    .reset_index(drop=True)
)

print("Component group and subgroup breakdown w/ decision:")
display(component_map_final.head(50))


# Summary of what was kept and what was dropped
component_group_summary = (
    comp_target.loc[comp_target['keep_for_component_model']]
    .groupby('component_group', dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values('rows', ascending=False)
    .reset_index(drop=True)
)

component_drop_summary = (
    comp_target.assign(
        drop_reason=comp_target['drop_reason'].fillna('KEEP')
    )
    .groupby('drop_reason', dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values('rows', ascending=False)
    .reset_index(drop=True)
)

print("\nSummary of kept groupings:")
display(component_group_summary)

print("\nSummary of dropped components:")
display(component_drop_summary)


# Check to see amount of remaining complaints are only about a single component group
component_model_rows = comp_target.loc[comp_target['keep_for_component_model']].copy()
case_group_counts = component_model_rows.groupby('odino')['component_group'].nunique()

single_case_ids = case_group_counts.loc[case_group_counts.eq(1)].index
multi_case_ids = case_group_counts.loc[case_group_counts.gt(1)].index

component_single_targets = (
    component_model_rows.loc[
        component_model_rows['odino'].isin(single_case_ids),
        ['odino', 'component_group']
    ]
    .drop_duplicates()
    .sort_values(['odino', 'component_group'])
    .groupby('odino', sort=True)
    .first()
    .reset_index()
)

component_single_case_summary = pd.DataFrame(
    [
        {
            'metric': 'final_component_groups',
            'value': int(component_group_summary['component_group'].nunique())
        },
        {
            'metric': 'kept_component_rows',
            'value': int(len(component_model_rows))
        },
        {
            'metric': 'kept_complaint_cases',
            'value': int(component_model_rows['odino'].nunique())
        },
        {
            'metric': 'single_label_cases',
            'value': int(len(single_case_ids))
        },
        {
            'metric': 'multi_label_cases',
            'value': int(len(multi_case_ids))
        },
        {
            'metric': 'single_label_case_share_pct',
            'value': round(float(len(single_case_ids) / component_model_rows['odino'].nunique() * 100), 2)
        }
    ]
)

print("\nShare of complaints that are single component and multi component:")
display(component_single_case_summary)

print("\nSample of some single component rows:")
display(component_single_targets.head(10))

## 10. Decision Queue

Not every question needs to be solved in the same pass. This queue explicity states what has been decided for each project task and what needs to be done next to keep track.

In [ ]:
decision_queue = pd.DataFrame([
    {
        'topic': 'component grouping',
        'current_position': 'component model table is ready but the cleaned component target is not finalized',
        'next_check': 'review unique raw component labels and build the first component group map'
    },
    {
        'topic': 'severity baseline feature list',
        'current_position': 'severity unit, label, and excluded leakage-prone fields are now settled',
        'next_check': 'freeze the first feature list before moving the cleaner into src'
    },
    {
        'topic': 'component baseline feature list',
        'current_position': 'component table is row-ready and severity-adjacent fields are excluded from the first pass',
        'next_check': 'confirm whether any dealer fields or date-derived fields belong in the first component baseline'
    },
    {
        'topic': 'src pipeline handoff',
        'current_position': 'notebook logic is close to final for the shared cleaning layer',
        'next_check': 'port the settled rules and first-pass model tables into a reproducible src module'
    }
])

display(decision_queue)

## 11. Scratchpad

This is a safe place for short checks and one-off comparisons without disrupting the main notebook flow.

In [ ]:
# Example checks
# compare_view(flag_df['flag_speed_high'], review_cols, n=25)
# candidate_df.loc[candidate_df['prod_type'] == 'V', ['maketxt', 'modeltxt', 'compdesc']].head(20)
# candidate_df.groupby('prod_type').size().sort_values(ascending=False)
# candidate_df.loc[flag_df['flag_miles_high'], ['maketxt', 'modeltxt', 'miles', 'cdescr']].head(20)
